Loading ML models and Historical CSV (for real lag calculation)...


d:\CA_content\Python\KissanSathi\krishisarthi-api\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


SESSION_FARMER_ID  : 740e8268-730f-4bf6-bc2d-540e7d3325e5
MANDI_JSON_PATH    : D:\CA_content\Python\KissanSathi\krishisarthi-api\data\mandi_master.json
DB_URL             : SET
Kharif XGB Model   : 700 estimators | 32 features
Rabi   XGB Model   : 300 estimators | 33 features
Historical Data    : 7841 rows loaded for Lags
✅ LangGraph compiled | Kharif + Rabi models ready

🌾 KisanSaathi | Farmer: 740e8268-730f-4bf6-bc2d-540e7d3325e5
Type 'bye' to exit

❌ ERROR: Error code: 400 - {'error': {'message': 'tool call validation failed: parameters for tool get_weather did not match schema: errors: [`/days`: expected integer, but got string]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=get_weather>{"days": "3"}</function>'}}
❌ ERROR: Error code: 400 - {'error': {'message': 'tool call validation failed: parameters for tool get_weather did not match schema: errors: [`/days`: expected integer, but got string]', 'type': 'invalid_request_error', 'code

Loading ML models and Historical CSV...


d:\CA_content\Python\KissanSathi\krishisarthi-api\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
<unknown>:1: SyntaxWarning: invalid escape sequence '\C'


Kharif XGB : 700 estimators | 32 features
Rabi   XGB : 300 estimators | 33 features
CSV Rows   : 7841
✅ LangGraph compiled | Tools: list_mandis, fetch_crop_price, get_weather, get_crop_advice

🌾 KisanSaathi | Farmer: 740e8268-730f-4bf6-bc2d-540e7d3325e5
Type 'bye' to exit


🤖 KisanSaathi: Namaste! KisanSaathi mein aapka swagat hai! Aapke liye kya kar sakta hoon? Aapki kisi fasal ki price jaanna chahte hain, mausam ke baare mein jaanna chahte hain, ya phir apni fasal ki advice lena chahte hain?


📋 [TOOL] list_mandis(crop='wheat' → 'wheat')

📋 [TOOL] list_mandis(crop='wheat' → 'wheat')

🤖 KisanSaathi: Kripaya apni pasand ki mandi ka naam ya number batayein, jisse main aapko wheat ki current price bata sakun.


🤖 KisanSaathi: Agra, Uttar Pradesh ke APMCs (wheat ke liye):
  1. Achnera APMC
  2. Agra APMC
  3. Fatehabad APMC
  4. Fatehpur Sikri APMC
  5. Jagnair APMC
  6. Jarar APMC

Inme se kaunsi mandi ka bhav jaanna chahte hain? (number ya naam batayein)


💰 [TOOL] fetch_crop_price('Jar

In [ ]:
import os, json, httpx, asyncpg, asyncio, joblib, time
import numpy as np
import pandas as pd
from typing import Annotated, List
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from dotenv import load_dotenv

# ══════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════
load_dotenv()
SESSION_FARMER_ID = os.getenv("SESSION_FARMER_ID")
MANDI_JSON_PATH   = os.getenv("MANDI_JSON_PATH")
MODEL_DIR         = os.getenv("MODEL_DIR_ABS")
AGMARKNET_URL     = os.getenv("AGMARKNET_URL")
OPEN_METEO_URL    = os.getenv("OPEN_METEO_URL")
DB_URL            = os.getenv("DATABASE_URL")
CSV_PATH          = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\Dataset\KrishiTwin_Final_Engineered.csv"

print("Loading ML models and Historical CSV...")
_xgb_kharif      = joblib.load(os.path.join(MODEL_DIR, "krishi_kharif_xgb_final.pkl"))
_xgb_rabi        = joblib.load(os.path.join(MODEL_DIR, "krishi_rabi_xgb_final.pkl"))
_kharif_features = joblib.load(os.path.join(MODEL_DIR, "kharif_feature_list.pkl"))
_rabi_features   = joblib.load(os.path.join(MODEL_DIR, "rabi_feature_list.pkl"))
_le_crop         = joblib.load(os.path.join(MODEL_DIR, "crop_encoder.pkl"))
_le_state        = joblib.load(os.path.join(MODEL_DIR, "state_encoder.pkl"))

hist_df = pd.read_csv(CSV_PATH)
hist_df.columns   = [c.replace('..', '.').strip() for c in hist_df.columns]
hist_df['dist_name']  = hist_df['dist_name'].str.lower().str.strip()
hist_df['State Name'] = hist_df['State Name'].str.lower().str.strip()

print(f"Kharif XGB : {_xgb_kharif.n_estimators} estimators | {len(_kharif_features)} features")
print(f"Rabi   XGB : {_xgb_rabi.n_estimators} estimators | {len(_rabi_features)} features")
print(f"CSV Rows   : {len(hist_df)}")

# ══════════════════════════════════════════
#  CHANGE 1 — Hinglish → English Crop Map
# ══════════════════════════════════════════
HINGLISH_CROP_MAP = {
    # Cereals
    "gehu": "wheat", "gehun": "wheat", "wheat": "wheat",
    "chawal": "rice", "dhan": "rice",  "rice": "rice",
    "makka": "maize", "maize": "maize", "corn": "maize",
    "bajra": "pearl millet", "bajra millet": "pearl millet",
    "jowar": "sorghum", "jwar": "sorghum",
    "jau": "barley", "barley": "barley",
    # Pulses
    "chana": "chickpea", "chane": "chickpea", "chickpea": "chickpea",
    "arhar": "pigeonpea", "tur": "pigeonpea", "toor": "pigeonpea",
    "masoor": "lentil", "masur": "lentil", "lentil": "lentil",
    "mung": "moong", "moong": "moong",
    "urad": "black gram",
    # Oilseeds
    "sarson": "mustard", "rai": "mustard", "mustard": "mustard",
    "mungfali": "groundnut", "moongphali": "groundnut", "groundnut": "groundnut",
    "soybean": "soyabean", "soya": "soyabean",
    "til": "sesamum", "sesame": "sesamum",
    "sunflower": "sunflower", "surajmukhi": "sunflower",
    # Cash crops
    "ganna": "sugarcane", "ganne": "sugarcane", "sugarcane": "sugarcane",
    "kapas": "cotton", "cotton": "cotton",
    # Vegetables
    "pyaaj": "onion", "pyaz": "onion", "onion": "onion",
    "tamatar": "tomato", "tamaatar": "tomato", "tomato": "tomato",
    "aloo": "potato", "aaloo": "potato", "potato": "potato",
    "lahsun": "garlic", "lasun": "garlic", "garlic": "garlic",
    "adrak": "ginger", "ginger": "ginger",
    "mirch": "chilli", "lal mirch": "chilli", "chilli": "chilli",
    "dhaniya": "coriander", "coriander": "coriander",
}

def normalize_crop(raw: str) -> str:
    """Convert Hinglish crop name to English for API calls."""
    return HINGLISH_CROP_MAP.get(raw.lower().strip(), raw.strip())

# ══════════════════════════════════════════
#  CHANGE 2 — Kharif / Rabi Routing
# ══════════════════════════════════════════
KHARIF_CROP_KEYWORDS = ['RICE', 'PEARL MILLET', 'GROUNDNUT', 'SUGARCANE',
                         'MAIZE', 'COTTON', 'SOYABEAN', 'SESAMUM',
                         'KHARIF SORGHUM', 'FINGER MILLET']
RABI_CROP_KEYWORDS   = ['CHICKPEA', 'WHEAT', 'MUSTARD', 'LENTIL',
                         'BARLEY', 'LINSEED', 'SAFFLOWER', 'RABI SORGHUM',
                         'RAPESEED']

# Crops that historically do better as alternatives (used in medium/high risk suggestions)
KHARIF_ALT_CROPS = ['RICE YIELD (Kg per ha)', 'GROUNDNUT YIELD (Kg per ha)',
                     'PEARL MILLET YIELD (Kg per ha)']
RABI_ALT_CROPS   = ['WHEAT.YIELD.Kg.per.ha.', 'CHICKPEA YIELD (Kg per ha)']

def _is_kharif(crop_type: str) -> bool:
    cu = crop_type.upper()
    return any(k in cu for k in KHARIF_CROP_KEYWORDS)

def _current_season() -> str:
    """Returns 'kharif' (Jun–Oct) or 'rabi' (Nov–May) based on current month."""
    import datetime
    return "kharif" if datetime.datetime.now().month in [6,7,8,9,10] else "rabi"

BENCHMARK_YIELDS = {
    "RICE YIELD (Kg per ha)":         1872.1,
    "WHEAT.YIELD.Kg.per.ha.":         2087.2,
    "MAIZE.YIELD.Kg.per.ha.":         1880.7,
    "SUGARCANE YIELD (Kg per ha)":    5619.3,
    "COTTON.YIELD.Kg.per.ha.":         296.7,
    "PEARL MILLET YIELD (Kg per ha)": 1001.1,
    "CHICKPEA YIELD (Kg per ha)":      817.6,
    "GROUNDNUT YIELD (Kg per ha)":    1152.9,
}

# ══════════════════════════════════════════
#  CHANGE 3 — System Prompt (Fixed Tool Calling + Better Flows)
# ══════════════════════════════════════════
# ══════════════════════════════════════════
#  GROQ-OPTIMIZED SYSTEM PROMPT
# ══════════════════════════════════════════
SYSTEM_INSTRUCTIONS = f"""You are KisanSaathi, a helpful agricultural assistant for Indian farmers.
Farmer ID: {SESSION_FARMER_ID}

You have access to agricultural tools. You MUST use the native tool calling feature to use them.

══ PRICE FLOW ══
RULE: You MUST ALWAYS use the list mandis tool BEFORE the fetch crop price tool. No exceptions.
Even if the farmer already mentions a mandi name — still use the list mandis tool first.
CRITICAL: Always translate Hindi crop names to English before using the tool (e.g., Gehu -> Wheat, Chawal -> Rice).

STEP 1: Farmer mentions any crop.
        → Immediately use the list mandis tool ONCE.
        → CRITICAL RULE: You MUST copy the entire numbered list of mandis from the tool's result and show it to the farmer in your message. Do NOT summarize or skip the list!
        → Ask: "Upar di gayi list mein se kaunsa number ya mandi naam chahiye?"
        → STOP and wait for farmer reply.

STEP 2: Farmer picks a mandi.
        → Use the fetch crop price tool ONCE. Show result. STOP.

══ WEATHER FLOW ══
If the farmer asks about the weather, rain, or heat, use the get weather tool. STOP.

══ CROP ADVICE FLOW ══
If the farmer asks how to improve their score, yield, or get a loan, use the get crop advice tool.
The tool will return alternative options. Do NOT tell the farmer to do all options sequentially. Explain them as 'Aapke paas yeh vikalp (options) hain: Ya toh aap Option 1 kar sakte hain... ya phir Option 2 kar sakte hain'. Explain what to change and why. STOP.

══ CROP COMPARISON FLOW ══
If farmer asks "agar X ki jagah Y ugaau" or "X vs Y" or "Y try karun to kya hoga":
  → Use the get crop advice tool with mode="compare" and target_crop="Y".
  → target_crop should be the NEW crop farmer wants to try.

══ RULES ══
- Do NOT output XML or <function> tags. Use the standard JSON tool selector.
- Do NOT use parenthesis or underscores in your text when deciding to use a tool.
- Never use more than one tool per turn.
- Always respond in friendly Hinglish (Hindi + English mix).
- ERROR HANDLING: If a tool returns an error message (like DB Error, API Error, or Not Found), respond politely. Say something like: "Maafi chahunga kisan bhai, abhi kuch takneeki kharabi (technical issue) ke kaaran main yeh jaankari nahi la paa raha hoon. Kripya thodi der baad dobara koshish karein."
"""

# ══════════════════════════════════════════
#  TOOL 1 — list_mandis (Hinglish crop fix)
# ══════════════════════════════════════════
@tool
async def list_mandis(crop: str) -> str:
    """Lists APMC mandis in farmer's district for a given crop.
    Crop name can be in Hindi/Hinglish — it will be translated automatically."""
    # CHANGE: normalize hinglish crop before API call
    crop_en = normalize_crop(crop)
    print(f"\n📋 [TOOL] list_mandis(crop='{crop}' → '{crop_en}')")
    try:
        conn = await asyncpg.connect(DB_URL)
        row  = await conn.fetchrow(
            "SELECT state_name, dist_name FROM farmers WHERE id = $1", SESSION_FARMER_ID)
        await conn.close()
    except Exception as e: return f"❌ DB Error: {e}"

    if not row: return "❌ Farmer not found."
    state, district = row["state_name"], row["dist_name"]

    try:
        with open(MANDI_JSON_PATH, "r", encoding="utf-8") as f:
            mandi_data = json.load(f)
    except FileNotFoundError: return "❌ Mandi master JSON not found."

    mandis = []
    for s_key, districts in mandi_data.items():
        if s_key.lower() == state.lower():
            for d_key, m_list in districts.items():
                if d_key.lower() == district.lower():
                    mandis = m_list
                    break

    if not mandis: return f"⚠️ {district} ({state}) ke liye koi mandi data nahi hai."
    mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(mandis))
    
    result = f"📍 {district}, {state} ke APMCs ({crop} ke liye):\n{mandi_list}"
    print(f"\n{result}\n")  # <--- THIS MAKES IT VISIBLE ON YOUR SCREEN
    
    return f"{result}\n\nKaunsi mandi ka bhav chahiye?"

# ══════════════════════════════════════════
#  TOOL 2 — fetch_crop_price (Hinglish fix)
# ══════════════════════════════════════════
@tool
async def fetch_crop_price(mandi_name: str, crop: str) -> str:
    """Fetches live price for a crop at a specific APMC mandi."""
    crop_en = normalize_crop(crop)
    print(f"\n💰 [TOOL] fetch_crop_price('{mandi_name}', '{crop}' → '{crop_en}')")
    try:
        conn = await asyncpg.connect(DB_URL)
        row  = await conn.fetchrow(
            "SELECT state_name FROM farmers WHERE id = $1", SESSION_FARMER_ID)
        await conn.close()
    except Exception as e: return f"❌ DB Error: {e}"

    if not row: return "❌ Farmer not found."
    clean_name = mandi_name.replace("APMC", "").strip()
    params = {
        "api-key": os.getenv("AGMARKNET_API_KEY"),
        "format": "json",
        "filters[state]": row["state_name"],
        "filters[market]": clean_name,
        "filters[commodity]": crop_en,
    }
    async with httpx.AsyncClient() as client:
        resp = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
    records = resp.json().get("records", [])
    if not records:
        return f"⚠️ Aaj {crop_en} ka data {clean_name} mandi mein available nahi hai."
    r = records[0]
    return (f"✅ Live Mandi Price\n"
            f"   Fasal  : {r['commodity']}\n"
            f"   Mandi  : {r['market']}\n"
            f"   Price  : ₹{r['modal_price']} / Quintal\n"
            f"   Date   : {r['arrival_date']}")

# ══════════════════════════════════════════
#  TOOL 3 — get_weather (int fix)
# ══════════════════════════════════════════
@tool
async def get_weather(days: int = 3) -> str:
    """Fetches weather forecast for farmer's field. days must be an integer."""
    days = int(days)   # CHANGE: force cast — prevents string type errors from LLM
    print(f"\n🌤️ [TOOL] get_weather(days={days})")
    try:
        conn = await asyncpg.connect(DB_URL)
        row  = await conn.fetchrow(
            "SELECT field_name, city_name, center_lat, center_lon "
            "FROM farm_fields WHERE farmer_id = $1 LIMIT 1",
            SESSION_FARMER_ID)
        await conn.close()
    except Exception as e: return f"❌ DB Error: {e}"

    if not row: return "⚠️ Khet registered nahi hai."
    lat, lon = float(row["center_lat"]), float(row["center_lon"])

    params = {
        "latitude": lat, "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max",
        "forecast_days": min(days, 16),
        "timezone": "Asia/Kolkata",
    }
    async with httpx.AsyncClient() as client:
        resp = await client.get(OPEN_METEO_URL, params=params, timeout=15.0)
    daily = resp.json().get("daily", {})
    dates = daily.get("time", [])
    if not dates: return "⚠️ Mausam data nahi mila."

    lines = [f"🌤️ Mausam — {row['field_name']} ({lat:.3f}, {lon:.3f})"]
    for i in range(len(dates)):
        lines.append(
            f"  📅 {dates[i]} | "
            f"Temp: {daily['temperature_2m_min'][i]}°C – {daily['temperature_2m_max'][i]}°C | "
            f"Barish: {daily['precipitation_sum'][i] or 0} mm | "
            f"Hawa: {daily['windspeed_10m_max'][i] or 0} km/h"
        )
    return "\n".join(lines)


# ══════════════════════════════════════════
#  HELPERS — Health Score + Predict Yield
# ══════════════════════════════════════════
def _compute_health_score(predicted_yield, soil_health_score, npk, wdi,
                           irr_ratio, kharif_rain, kharif_temp,
                           rabi_temp, ndvi, crop_type) -> float:
    benchmark = BENCHMARK_YIELDS.get(crop_type, 2000.0)
    yield_sc  = min((predicted_yield / benchmark) * 100, 100) if benchmark > 0 else 50.0
    soil_base = min((soil_health_score / 200) * 100, 100)
    npk_pen   = min((abs(npk - 120) / 120) * 30, 30)
    soil_sc   = max(soil_base - npk_pen, 0)
    wdi_sc    = (1 - wdi) * 100
    irr_sc    = max((1 - abs(1 - irr_ratio)) * 100, 0)
    rain_sc   = min((kharif_rain / 600) * 100, 100)
    water_sc  = 0.5 * wdi_sc + 0.3 * irr_sc + 0.2 * rain_sc
    clim_sc   = max(100 - max(0, kharif_temp - 35) * 5 - max(0, rabi_temp - 25) * 5, 0)
    ndvi_sc   = min(ndvi * 125, 100) if ndvi is not None else min(yield_sc * 0.8, 100)
    return round(yield_sc*0.25 + soil_sc*0.20 + water_sc*0.25 + clim_sc*0.15 + ndvi_sc*0.15, 2)

def _predict_yield(inputs: dict) -> float:
    try:    crop_enc  = int(_le_crop.transform([inputs["Crop_Type"]])[0])
    except: crop_enc  = 0
    try:    state_enc = int(_le_state.transform([inputs["State_Name"]])[0])
    except: state_enc = 0

    kharif    = _is_kharif(inputs["Crop_Type"])
    mdl       = _xgb_kharif    if kharif else _xgb_rabi
    feat_list = _kharif_features if kharif else _rabi_features

    feat_dict = {
        "year":                            inputs.get("year", 2015),
        "State_Encoded":                   state_enc,
        "Crop_Encoded":                    crop_enc,
        "NPK_Intensity_KgHa":              inputs["NPK_Intensity_KgHa"],
        "Irrigation_Intensity_Ratio":      inputs["Irrigation_Intensity_Ratio"],
        "WDI":                             inputs["WDI"],
        "Kharif_Avg_MaxTemp":              inputs["Kharif_Avg_MaxTemp"],
        "Kharif_Total_Rain":               inputs["Kharif_Total_Rain"],
        "Rabi_Avg_MaxTemp":                inputs["Rabi_Avg_MaxTemp"],
        "District_Soil_Health_Score":      inputs["District_Soil_Health_Score"],
        "NPK_Intensity_KgHa_Lag1":         inputs["NPK_Lag1"],
        "NPK_Intensity_KgHa_Lag2":         inputs["NPK_Lag2"],
        "NPK_Intensity_KgHa_Lag3":         inputs["NPK_Lag3"],
        "Irrigation_Intensity_Ratio_Lag1": inputs["Irr_Lag1"],
        "Irrigation_Intensity_Ratio_Lag2": inputs["Irr_Lag2"],
        "Irrigation_Intensity_Ratio_Lag3": inputs["Irr_Lag3"],
        "WDI_Lag1":                        inputs["WDI_Lag1"],
        "WDI_Lag2":                        inputs["WDI_Lag2"],
        "WDI_Lag3":                        inputs["WDI_Lag3"],
        "Kharif_Avg_MaxTemp_Lag1":         inputs["KTemp_Lag1"],
        "Kharif_Avg_MaxTemp_Lag2":         inputs["KTemp_Lag2"],
        "Kharif_Avg_MaxTemp_Lag3":         inputs["KTemp_Lag3"],
        "Kharif_Total_Rain_Lag1":          inputs["KRain_Lag1"],
        "Kharif_Total_Rain_Lag2":          inputs["KRain_Lag2"],
        "Kharif_Total_Rain_Lag3":          inputs["KRain_Lag3"],
        "Rabi_Avg_MaxTemp_Lag1":           inputs["RTemp_Lag1"],
        "Rabi_Avg_MaxTemp_Lag2":           inputs["RTemp_Lag2"],
        "Rabi_Avg_MaxTemp_Lag3":           inputs["RTemp_Lag3"],
        "Kharif_Avg_MaxTemp_Delta1":  inputs["Kharif_Avg_MaxTemp"] - inputs["KTemp_Lag1"],
        "Kharif_Total_Rain_Delta1":   inputs["Kharif_Total_Rain"]  - inputs["KRain_Lag1"],
        "NPK_Intensity_KgHa_Delta1":  inputs["NPK_Intensity_KgHa"] - inputs["NPK_Lag1"],
        "Kharif_Avg_MaxTemp_Roll3": (inputs["KTemp_Lag1"]+inputs["KTemp_Lag2"]+inputs["KTemp_Lag3"]) / 3.0,
        "Kharif_Total_Rain_Roll3":  (inputs["KRain_Lag1"]+inputs["KRain_Lag2"]+inputs["KRain_Lag3"]) / 3.0,
    }
    feat_df  = pd.DataFrame([feat_dict])[feat_list]
    log_pred = float(mdl.predict(feat_df)[0])
    return float(np.expm1(min(max(log_pred, 0.0), 11.0)))

def score_band(s: float) -> str:
    if s >= 65: return "LOW RISK ✅ — Loan Eligible"
    if s >= 45: return "MEDIUM RISK ⚠️"
    return "HIGH RISK ❌"


# ══════════════════════════════════════════
#  TOOL 4 — get_crop_advice (mode-based)
# ══════════════════════════════════════════
# ── CHANGE: Add target_crop parameter ─────────────────────────────────────
@tool
async def get_crop_advice(mode: str = "advice", target_crop: str = "") -> str:
    """
    mode='score'    → current health score only.
    mode='advice'   → NPK + Irrigation what-if simulation.
    mode='cropswap' → alternative crops from district history.
    mode='compare'  → compare current crop vs target_crop specifically.
                      Requires target_crop e.g. target_crop='RICE YIELD (Kg per ha)'
    """
    start_time = time.perf_counter()
    print(f"\n🌱 [TOOL] get_crop_advice(mode='{mode}')")

    # ── DB Fetch ──────────────────────────────────────────────────────────
    try:
        conn   = await asyncpg.connect(DB_URL)
        farm   = await conn.fetchrow("""
            SELECT ff.id AS field_id, ff.city_name, ff.state_name,
                   f.state_name AS farmer_state, f.dist_name AS farmer_dist
            FROM farm_fields ff
            JOIN farmers f ON f.id = ff.farmer_id
            WHERE ff.farmer_id = $1
            ORDER BY ff.created_at DESC LIMIT 1
        """, SESSION_FARMER_ID)
        season = await conn.fetchrow("""
            SELECT crop_type, npk_input, irrigation_ratio, wdi_used,
                   ndvi_value, final_health_score, predicted_yield, year,
                   kharif_temp_used, kharif_rain_used, rabi_temp_used, soil_score_used
            FROM field_predictions
            WHERE field_id = $1
            ORDER BY year DESC LIMIT 1
        """, str(farm["field_id"]))
        await conn.close()
    except Exception as e: return f"❌ DB Error: {e}"

    if not season: return "⚠️ Koi prediction data nahi hai. Pehle field prediction karein."

    # ── CSV Lag Lookup ────────────────────────────────────────────────────
    dist_name = (farm["city_name"] or farm["farmer_dist"] or "").lower().strip()
    dist_hist = (hist_df[hist_df['dist_name'] == dist_name]
                 .sort_values('year', ascending=False)
                 .to_dict('records'))

    if len(dist_hist) < 3:
        return f"⚠️ '{dist_name}' ke liye CSV mein 3 saal ka data nahi mila."

    lag1, lag2, lag3 = dist_hist[0], dist_hist[1], dist_hist[2]

    # ── CHANGE: Year Fallback ─────────────────────────────────────────────
    requested_year  = int(season["year"] or 2026)
    latest_csv_year = int(lag1['year'])
    effective_year  = latest_csv_year if requested_year > latest_csv_year else requested_year
    if requested_year > latest_csv_year:
        print(f"   └─ ⚠️  Year {requested_year} → snapped to {effective_year} (CSV max)")

    def _v(val, fb): return float(val) if val is not None else fb

    crop_type  = season["crop_type"]
    is_kharif  = _is_kharif(crop_type)
    model_used = "Kharif" if is_kharif else "Rabi"
    print(f"   └─ Crop: {crop_type} | Model: {model_used} | Year: {effective_year}")
    print(f"   └─ Lags: {lag1['year']}, {lag2['year']}, {lag3['year']} ({dist_name})")

    base = {
        "State_Name":                 farm["state_name"] or farm["farmer_state"],
        "Crop_Type":                  crop_type,
        "year":                       effective_year,
        "NPK_Intensity_KgHa":         _v(season["npk_input"],         lag1['NPK_Intensity_KgHa']),
        "Irrigation_Intensity_Ratio": _v(season["irrigation_ratio"],   lag1['Irrigation_Intensity_Ratio']),
        "WDI":                        _v(season["wdi_used"],           lag1['WDI']),
        "Kharif_Avg_MaxTemp":         _v(season["kharif_temp_used"],   lag1['Kharif_Avg_MaxTemp']),
        "Kharif_Total_Rain":          _v(season["kharif_rain_used"],   lag1['Kharif_Total_Rain']),
        "Rabi_Avg_MaxTemp":           _v(season["rabi_temp_used"],     lag1['Rabi_Avg_MaxTemp']),
        "District_Soil_Health_Score": _v(season["soil_score_used"],    lag1['District_Soil_Health_Score']),
        "ndvi":                       season["ndvi_value"],
        "NPK_Lag1":   lag1['NPK_Intensity_KgHa'],   "NPK_Lag2":   lag2['NPK_Intensity_KgHa'],   "NPK_Lag3":   lag3['NPK_Intensity_KgHa'],
        "Irr_Lag1":   lag1['Irrigation_Intensity_Ratio'], "Irr_Lag2": lag2['Irrigation_Intensity_Ratio'], "Irr_Lag3": lag3['Irrigation_Intensity_Ratio'],
        "WDI_Lag1":   lag1['WDI'],   "WDI_Lag2":   lag2['WDI'],   "WDI_Lag3":   lag3['WDI'],
        "KTemp_Lag1": lag1['Kharif_Avg_MaxTemp'], "KTemp_Lag2": lag2['Kharif_Avg_MaxTemp'], "KTemp_Lag3": lag3['Kharif_Avg_MaxTemp'],
        "KRain_Lag1": lag1['Kharif_Total_Rain'],  "KRain_Lag2": lag2['Kharif_Total_Rain'],  "KRain_Lag3": lag3['Kharif_Total_Rain'],
        "RTemp_Lag1": lag1['Rabi_Avg_MaxTemp'],   "RTemp_Lag2": lag2['Rabi_Avg_MaxTemp'],   "RTemp_Lag3": lag3['Rabi_Avg_MaxTemp'],
    }

    current_yield = _v(season["predicted_yield"], _predict_yield(base))
    current_score = _v(season["final_health_score"],
                       _compute_health_score(current_yield,
                           base["District_Soil_Health_Score"], base["NPK_Intensity_KgHa"],
                           base["WDI"], base["Irrigation_Intensity_Ratio"],
                           base["Kharif_Total_Rain"], base["Kharif_Avg_MaxTemp"],
                           base["Rabi_Avg_MaxTemp"], base["ndvi"], crop_type))
    band = score_band(current_score)

    # ══════════════════════════════════════
    #  MODE: score — just return status
    # ══════════════════════════════════════
    if mode == "score":
        return (
            f"📊 CURRENT HEALTH SCORE\n"
            f"   Fasal      : {crop_type}\n"
            f"   District   : {dist_name}\n"
            f"   Yield      : {current_yield:.0f} kg/ha\n"
            f"   Score      : {current_score:.1f} / 100\n"
            f"   Risk Band  : {band}\n"
            f"   Data Basis : CSV lags {lag1['year']}, {lag2['year']}, {lag3['year']}\n"
            f"   Model Used : {model_used} XGBoost\n"
            + (f"\n✅ Aap loan ke liye eligible hain!" if current_score >= 65
               else f"\n⚠️  Score thoda aur badhana hoga loan eligibility ke liye (target: 65+).")
        )

    # ══════════════════════════════════════
    #  MODE: cropswap — alternative crops
    # ══════════════════════════════════════
    if mode == "cropswap":
        season_now  = _current_season()
        alt_cols    = KHARIF_ALT_CROPS if season_now == "kharif" else RABI_ALT_CROPS
        # Filter to columns that actually exist in CSV
        alt_cols    = [c for c in alt_cols if c in hist_df.columns]
        dist_yields = hist_df[hist_df['dist_name'] == dist_name].tail(5)

        lines = [
            f"🔄 ALTERNATIVE CROPS — {dist_name.title()} district | Season: {season_now.upper()}",
            f"   (Based on last 5 years of district CSV data)\n"
        ]
        found_any = False
        for col in alt_cols:
            if col not in dist_yields.columns: continue
            vals = dist_yields[col].dropna()
            if len(vals) == 0: continue
            avg  = vals.mean()
            benchmark = BENCHMARK_YIELDS.get(col, 1500.0)
            perf = (avg / benchmark) * 100 if benchmark > 0 else 50
            # Simulate score with current inputs but this crop
            alt_base   = {**base, "Crop_Type": col}
            try:
                alt_yield = _predict_yield(alt_base)
                alt_score = _compute_health_score(
                    alt_yield, base["District_Soil_Health_Score"],
                    base["NPK_Intensity_KgHa"], base["WDI"],
                    base["Irrigation_Intensity_Ratio"], base["Kharif_Total_Rain"],
                    base["Kharif_Avg_MaxTemp"], base["Rabi_Avg_MaxTemp"],
                    base["ndvi"], col)
                lines.append(
                    f"  🌾 {col}\n"
                    f"     District 5yr Avg Yield : {avg:.0f} kg/ha\n"
                    f"     Model Predicted Yield  : {alt_yield:.0f} kg/ha\n"
                    f"     Simulated Score        : {alt_score:.1f}/100 — {score_band(alt_score)}\n"
                    f"     Aapke current NPK ({base['NPK_Intensity_KgHa']:.0f} kg/ha) "
                    f"aur Irrigation ({base['Irrigation_Intensity_Ratio']:.0%}) par yeh fasal "
                    f"{'behtar' if alt_score > current_score else 'similar'} performance degi.\n"
                )
                found_any = True
            except Exception:
                continue

        if not found_any:
            return f"⚠️ {dist_name} ke liye alternate crop simulation nahi ho saki."
        lines.append(f"📌 Current fasal ({crop_type}) ka score: {current_score:.1f}/100 ({band})")
        return "\n".join(lines)
    # ══════════════════════════════════════
    #  MODE: compare — specific crop vs crop
    # ══════════════════════════════════════
    if mode == "compare":
        if not target_crop:
            return "⚠️ target_crop parameter missing. Please specify which crop to compare."

        # Normalize hinglish input
        target_crop_en = normalize_crop(target_crop)

        # Find matching column in CSV
        matched_col = None
        for col in hist_df.columns:
            if target_crop_en.upper() in col.upper():
                matched_col = col
                break

        if not matched_col:
            return (f"⚠️ '{target_crop}' ({target_crop_en}) ka data CSV mein nahi mila.\n"
                    f"   Available crops: {[c for c in BENCHMARK_YIELDS.keys()]}")

        # Current crop simulation
        current_yield = _v(season["predicted_yield"], _predict_yield(base))
        current_score = _v(season["final_health_score"],
                           _compute_health_score(current_yield,
                               base["District_Soil_Health_Score"], base["NPK_Intensity_KgHa"],
                               base["WDI"], base["Irrigation_Intensity_Ratio"],
                               base["Kharif_Total_Rain"], base["Kharif_Avg_MaxTemp"],
                               base["Rabi_Avg_MaxTemp"], base["ndvi"], crop_type))

        # Target crop simulation — same inputs, different crop
        target_base  = {**base, "Crop_Type": matched_col}
        try:
            target_yield = _predict_yield(target_base)
            target_score = _compute_health_score(
                target_yield, base["District_Soil_Health_Score"],
                base["NPK_Intensity_KgHa"], base["WDI"],
                base["Irrigation_Intensity_Ratio"], base["Kharif_Total_Rain"],
                base["Kharif_Avg_MaxTemp"], base["Rabi_Avg_MaxTemp"],
                base["ndvi"], matched_col)
        except Exception as e:
            return f"❌ {matched_col} ke liye simulation fail hui: {e}"

        # District historical average for target crop
        dist_avg_target = (hist_df[hist_df['dist_name'] == dist_name][matched_col]
                           .dropna().tail(5).mean())
        dist_avg_current_col = None
        for col in hist_df.columns:
            if crop_type.upper().split()[0] in col.upper():
                dist_avg_current_col = col
                break
        dist_avg_current = (hist_df[hist_df['dist_name'] == dist_name][dist_avg_current_col]
                            .dropna().tail(5).mean()
                            if dist_avg_current_col else current_yield)

        yield_diff  = target_yield - current_yield
        score_diff  = target_score - current_score
        yield_arrow = "📈" if yield_diff > 0 else "📉"
        score_arrow = "⬆️" if score_diff > 0 else "⬇️"

        return "\n".join([
            f"🔄 CROP COMPARISON — {dist_name.title()} District",
            f"   Same NPK ({base['NPK_Intensity_KgHa']:.0f} kg/ha) aur "
            f"Irrigation ({base['Irrigation_Intensity_Ratio']:.0%}) par:\n",

            f"  📌 Current  : {crop_type}",
            f"     Yield    : {current_yield:.0f} kg/ha",
            f"     Score    : {current_score:.1f}/100 — {score_band(current_score)}",
            f"     District 5yr Avg: {dist_avg_current:.0f} kg/ha\n",

            f"  🌾 Switch To: {matched_col}",
            f"     Yield    : {target_yield:.0f} kg/ha  {yield_arrow} ({yield_diff:+.0f} kg/ha)",
            f"     Score    : {target_score:.1f}/100 — {score_band(target_score)}  "
            f"{score_arrow} ({score_diff:+.1f} points)",
            f"     District 5yr Avg: {dist_avg_target:.0f} kg/ha\n",

            f"  📊 Verdict  : "
            + ("Is badlaav se aapka yield aur score DONO behtar honge." if yield_diff > 0 and score_diff > 0
               else "Yield badhega lekin score thoda kam ho sakta hai." if yield_diff > 0
               else "Yeh fasal aapke district mein relatively kam perform karti hai."
                    " Badlaav sochkar karein."),

            "",
            f"  ⚠️  Note: Yeh simulation aapke CURRENT inputs par hai. "
            f"{matched_col.split()[0]} ke liye alag NPK/Irrigation optimal ho sakta hai."
            + ("\n   [CROPSWAP_RECOMMENDED]" if target_score < 65 else "")
        ])

    # ══════════════════════════════════════
    #  MODE: advice — full what-if
    # ══════════════════════════════════════
    orig_npk = base["NPK_Intensity_KgHa"]
    orig_irr = base["Irrigation_Intensity_Ratio"]

    # NPK — find single best
    best_npk, best_npk_score = None, current_score
    for npk_val in [60, 80, 100, 120, 140, 160]:
        if abs(npk_val - orig_npk) < 5: continue
        m         = {**base, "NPK_Intensity_KgHa": float(npk_val)}
        ny        = _predict_yield(m)
        ns        = _compute_health_score(ny, m["District_Soil_Health_Score"], npk_val,
                        m["WDI"], m["Irrigation_Intensity_Ratio"], m["Kharif_Total_Rain"],
                        m["Kharif_Avg_MaxTemp"], m["Rabi_Avg_MaxTemp"], m["ndvi"], m["Crop_Type"])
        if ns > best_npk_score and (ns - current_score > 0.5):
            best_npk_score = ns
            best_npk = {"npk_val": npk_val, "new_yield": ny, "new_score": ns,
                        "gain": ns - current_score}

    # Irrigation — find single best
    best_irr, best_irr_score = None, current_score
    for irr_val in [0.3, 0.5, 0.7, 0.85, 1.0]:
        if abs(irr_val - orig_irr) < 0.05: continue
        m  = {**base, "Irrigation_Intensity_Ratio": float(irr_val)}
        ny = _predict_yield(m)
        ns = _compute_health_score(ny, m["District_Soil_Health_Score"], m["NPK_Intensity_KgHa"],
                m["WDI"], irr_val, m["Kharif_Total_Rain"],
                m["Kharif_Avg_MaxTemp"], m["Rabi_Avg_MaxTemp"], m["ndvi"], m["Crop_Type"])
        if ns > best_irr_score and (ns - current_score > 0.5):
            best_irr_score = ns
            best_irr = {"irr_val": irr_val, "new_yield": ny, "new_score": ns,
                        "gain": ns - current_score}

    end_time = time.perf_counter()
    print(f"   └─ ⏱️  {model_used} | {end_time - start_time:.4f}s")

    # ── Build response ────────────────────────────────────────────────────
    lines = [
        f"📊 CURRENT STATUS  [{model_used} Model]",
        f"   Fasal     : {crop_type}",
        f"   District  : {dist_name}",
        f"   Yield     : {current_yield:.0f} kg/ha",
        f"   Score     : {current_score:.1f} / 100  →  {band}",
        f"   Data Basis: CSV lags {lag1['year']}, {lag2['year']}, {lag3['year']}",
        f"   Year Used : {effective_year}"
        + (f"  ⚠️  (requested {requested_year}, snapped to CSV max)" if effective_year != requested_year else ""),
        ""
    ]

    # Low risk — special message
    if current_score >= 65 and not best_npk and not best_irr:
        lines += [
            "✅ LOAN ELIGIBLE — Aapki fasal pehle se hi kaafi achi hai!",
            f"   Score {current_score:.1f}/100 — Aap abhi loan ke liye eligible hain.",
            "",
            "💡 Koi bhi badlaav zaroori nahi hai. Lekin agar aap aur improve karna",
            "   chahte hain ya doosri fasal try karna chahte hain, toh batayein.",
            "   [CROPSWAP_ELIGIBLE]"
        ]
        return "\n".join(lines)

    if not best_npk and not best_irr:
        lines += [
            "✅ Aapki farming already optimal hai! Koi improvement nahi mila.",
            "   [CROPSWAP_ELIGIBLE]"
        ]
        return "\n".join(lines)

    lines.append("💡 BEST IMPROVEMENT OPTIONS — Aap inme se EK option choose karein:\n")
    option_num = 1

    # ── CHANGE: Irrigation advice — realistic framing ─────────────────────
    if best_irr:
        irr_val    = best_irr["irr_val"]
        irr_change = "badhani" if irr_val > orig_irr else "ghata"
        irr_note   = (
            "Hum jaante hain ki hamesha 100% sinchai possible nahi hoti."
            if irr_val >= 0.85 else
            f"Aapke {dist_name} district ki soil health aur pichle saalon ke data ke hisaab se,"
        )
        irr_impact = (
            f"agar aap sinchai kam karke {irr_val:.0%} par laayen to bhi yield badh sakti hai."
            if irr_val < orig_irr else
            f"agar aap poore khet mein sinchai {irr_val:.0%} tak rakh saken to yield behtar hogi."
        )
        lines += [
            f"  Option {option_num}: Sinchai (Irrigation) Badlaav",
            f"   {irr_note} {irr_impact}",
            f"   → Irrigation: {orig_irr:.0%}  →  {irr_val:.0%}",
            f"   → Nayi Yield : {best_irr['new_yield']:.0f} kg/ha",
            f"   → Naya Score : {best_irr['new_score']:.1f}/100  →  {score_band(best_irr['new_score'])}",
            f"   → Gain       : +{best_irr['gain']:.1f} score points",
            ""
        ]
        option_num += 1

    # ── NPK advice ────────────────────────────────────────────────────────
    if best_npk:
        npk_val    = best_npk["npk_val"]
        npk_reason = (
            "Zyada chemical khad mitti ki sehat ko nuksan pahuncha raha hai."
            if npk_val < orig_npk else
            "Sahi poshan milne se paudhe zyada healthy honge aur yield badhegi."
        )
        lines += [
            f"  Option {option_num}: Fertilizer (NPK) Badlaav",
            f"   Aapke district ke pichle data ke hisaab se yeh NPK level zyada effective hai.",
            f"   → NPK      : {orig_npk:.0f}  →  {npk_val} kg/ha",
            f"   → Reason   : {npk_reason}",
            f"   → Nayi Yield: {best_npk['new_yield']:.0f} kg/ha",
            f"   → Naya Score: {best_npk['new_score']:.1f}/100  →  {score_band(best_npk['new_score'])}",
            f"   → Gain      : +{best_npk['gain']:.1f} score points",
            ""
        ]

    # Medium/High risk — add cropswap signal for LLM
    if current_score < 65:
        lines += [
            "⚠️  [MEDIUM_HIGH_RISK] — Aapka score abhi bhi loan threshold (65) se neeche hai.",
            "   District history mein kuch aur faslein better perform karti hain.",
            "   [CROPSWAP_RECOMMENDED]"
        ]

    return "\n".join(lines)


# ══════════════════════════════════════════
#  STATE & ROUTING
# ══════════════════════════════════════════
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

llm_model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
).bind_tools(
    [list_mandis, fetch_crop_price, get_weather, get_crop_advice],
    parallel_tool_calls=False,
    tool_choice="auto",
)

async def call_model(state: AgentState) -> dict:
    return {"messages": [await llm_model.ainvoke(
        [SystemMessage(content=SYSTEM_INSTRUCTIONS)] + state["messages"])]}

async def call_tool(state: AgentState) -> dict:
    last_msg = state["messages"][-1]
    results  = []
    for tc in last_msg.tool_calls:
        name, args = tc["name"], tc["args"]
        # CHANGE: safe int cast for days before dispatch
        if name == "get_weather" and "days" in args:
            args["days"] = int(args["days"])
        if   name == "list_mandis":       res = await list_mandis.ainvoke(args)
        elif name == "fetch_crop_price":  res = await fetch_crop_price.ainvoke(args)
        elif name == "get_weather":       res = await get_weather.ainvoke(args)
        elif name == "get_crop_advice":   res = await get_crop_advice.ainvoke(args)
        else:                             res = f"❌ Unknown tool: {name}"
        results.append(ToolMessage(tool_call_id=tc["id"], content=str(res)))
    return {"messages": results}

def router(state: AgentState) -> str:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END

graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", call_tool)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", router, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")
app = graph.compile()
print("✅ LangGraph compiled | Tools: list_mandis, fetch_crop_price, get_weather, get_crop_advice")

# ══════════════════════════════════════════
#  CHAT
# ══════════════════════════════════════════
async def chat_with_farmer():
    print(f"\n🌾 KisanSaathi | Farmer: {SESSION_FARMER_ID}\nType 'bye' to exit\n")
    state = {"messages": []}
    while True:
        try:    user_input = input("👨‍🌾 Farmer: ").strip()
        except EOFError: break
        if not user_input: continue
        if user_input.lower() in ("quit", "exit", "bye"): break

        state["messages"].append(HumanMessage(content=user_input))
        try:
            result = await app.ainvoke(state, config={"recursion_limit": 10})
            state["messages"] = result["messages"]
        except Exception as e:
            print(f"❌ ERROR: {e}")
            continue

        for msg in reversed(state["messages"]):
            if isinstance(msg, AIMessage) and not msg.tool_calls:
                print(f"\n🤖 KisanSaathi: {msg.content}\n")
                break

await chat_with_farmer()

Loading ML models and Historical CSV...
Kharif XGB : 700 estimators | 32 features
Rabi   XGB : 300 estimators | 33 features
CSV Rows   : 7841


d:\CA_content\Python\KissanSathi\krishisarthi-api\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✅ LangGraph compiled | Tools: list_mandis, fetch_crop_price, get_weather, get_crop_advice

🌾 KisanSaathi | Farmer: 740e8268-730f-4bf6-bc2d-540e7d3325e5
Type 'bye' to exit

